# **Support Vector Machines**

In [64]:
%matplotlib widget

import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import random
import math
from cvxopt import matrix, solvers

EPS = math.e

In [65]:
df: pd.DataFrame = pd.read_csv("iris.csv")
display(df.head())
display(df.shape)

,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa


(150, 5)

In [66]:
display(df['variety'].unique())

array(['Setosa', 'Versicolor', 'Virginica'], dtype=object)

In [67]:
v1 = 'Setosa'
v2 = 'Versicolor'
dataset = df.loc[df['variety'].isin([v1, v2])].replace({'variety':{v1: "1", v2: "-1"}})
# display(dataset.shape)
#display(dataset.head())
#display(dataset['variety'].unique())

xx = dataset.loc[:, df.columns != 'variety'].to_numpy().astype(float)
yy = np.array([label for label in dataset['variety']]).astype(float)
print(xx)
display(yy)

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]
 [5.4 3.9 1.7 0.4]
 [4.6 3.4 1.4 0.3]
 [5.  3.4 1.5 0.2]
 [4.4 2.9 1.4 0.2]
 [4.9 3.1 1.5 0.1]
 [5.4 3.7 1.5 0.2]
 [4.8 3.4 1.6 0.2]
 [4.8 3.  1.4 0.1]
 [4.3 3.  1.1 0.1]
 [5.8 4.  1.2 0.2]
 [5.7 4.4 1.5 0.4]
 [5.4 3.9 1.3 0.4]
 [5.1 3.5 1.4 0.3]
 [5.7 3.8 1.7 0.3]
 [5.1 3.8 1.5 0.3]
 [5.4 3.4 1.7 0.2]
 [5.1 3.7 1.5 0.4]
 [4.6 3.6 1.  0.2]
 [5.1 3.3 1.7 0.5]
 [4.8 3.4 1.9 0.2]
 [5.  3.  1.6 0.2]
 [5.  3.4 1.6 0.4]
 [5.2 3.5 1.5 0.2]
 [5.2 3.4 1.4 0.2]
 [4.7 3.2 1.6 0.2]
 [4.8 3.1 1.6 0.2]
 [5.4 3.4 1.5 0.4]
 [5.2 4.1 1.5 0.1]
 [5.5 4.2 1.4 0.2]
 [4.9 3.1 1.5 0.2]
 [5.  3.2 1.2 0.2]
 [5.5 3.5 1.3 0.2]
 [4.9 3.6 1.4 0.1]
 [4.4 3.  1.3 0.2]
 [5.1 3.4 1.5 0.2]
 [5.  3.5 1.3 0.3]
 [4.5 2.3 1.3 0.3]
 [4.4 3.2 1.3 0.2]
 [5.  3.5 1.6 0.6]
 [5.1 3.8 1.9 0.4]
 [4.8 3.  1.4 0.3]
 [5.1 3.8 1.6 0.2]
 [4.6 3.2 1.4 0.2]
 [5.3 3.7 1.5 0.2]
 [5.  3.3 1.4 0.2]
 [7.  3.2 4.7 1.4]
 [6.4 3.2 4.5 1.5]
 [6.9 3.1 4.

array([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1.])

In [68]:
# 70% for train and 30% test
m, = yy.shape
idx = np.array(np.arange(m))
include_idx = set(np.random.choice(idx, size = int(0.7*m), replace=False))
mask = np.array([(i in include_idx) for i in range(m)])

trainX = xx[mask]
testX = xx[~mask]

trainY = yy[mask]
testY = yy[~mask]

display(trainX.shape, testX.shape)
display(trainY.shape, testY.shape)

(70, 4)

(30, 4)

(70,)

(30,)

# **Multiplicadores de Lagrange**
$\frac{\partial f(x)}{ \partial x} = λ \frac{g( \partial x)}{ \partial x}$

Hallar los valores de $λ_i$ para cada elemento de entrenamiento $X_i$. 

El código ***GetLambda***  debe retornar un vector al cual denominaremos lambda, de modo que
  $lambda[i]$ será $0$, si el elemento $X[i]$ no tiene intercesión con ninguna de las rectas
  $XW^t + b >=1$ o $XW^t + b >=0$

- **Nota: Puede buscar en internet la forma de como hallar lambda.**

In [69]:
def GetLambda(X, Y):
  m, n = X.shape
  P = np.empty((m, m))
  for i in range(m):
    for j in range(m):
      P[i, j] = Y[i]*Y[j]*np.dot(X[i], X[j])
  q = -np.ones((m, 1))
  G = -np.eye(m)
  h = np.zeros((m, 1))
  A = Y.reshape((1, m))
  b = np.zeros((1, 1))

  P = matrix(P)
  q = matrix(q)
  G = matrix(G)
  h = matrix(h)
  A = matrix(A.astype('double'))
  b = matrix(b)

  #Setting solver parameters (change default to decrease tolerance) 
  solvers.options['show_progress'] = False
  solvers.options['abstol'] = 1e-10
  solvers.options['reltol'] = 1e-10
  solvers.options['feastol'] = 1e-10

  sol = solvers.qp(P, q, G, h, A, b)
  return np.array(sol['x']).reshape(m)
  

## 2. Cálculo de los pesos W
$W_j = \sum_{i=0}^n \lambda_iy_ix_{ij}$  

Donde: λ_i es el i-esimo multiplicador de lagrange, W_j es el W-esimo peso y x_{ij} es el valor de la característica $j$ del objeto de entrenamiento $i-esimo$ y $y_i$ es la salida esperada (1 o - 1) del objeto $i$.

Recuerde la sumatoria solo recorre todos los elementos para los cuales el valor del multiplicador de lagrange $λ_i$ es diferente de 0.

In [70]:
def Get_W(X,Y, lambd):
    n, = Y.shape
    r, c = X.shape
    return np.array([np.sum([lambd*X[i,j]*Y[i] for i in range(n)]) for j in range(c)])

## Cálculo de b

$$XW^t + b = 0$$

$$b = - ∑_{i=0}^n X_iW^t$$

Donde $X_i$ es un vector $k$ dimensional y representa el objeto $i-esimo$ de entrenamiento y $k$ el número de características del objeto.

In [71]:
def Get_b(X,W):
    n, = W.shape
    wT = np.transpose(W)
    return -1 * np.sum([xi * wT for xi in X])/n

In [72]:
def Get_b_2(Y, X,W):
    n, = W.shape
    wT = np.transpose(W)
    return 1 * np.sum([xi * wT for xi in X])/n

## Etapa de Testing

Para esta estapa solo se debe calcular

$f(X_j) = X_jW^t + b$

Pero dado que ya hemos calculado el valor de los parámetros $W$ y $b$, entonces remplazando tenemos

$f(X_j) = \sum_{i=0}^n \lambda_iy_i<X_{i},X_{j}> + b$

Donde: $X_i$ i-esimo  es el vector de entrenamiento y $X_j$ es el nuevo vector que pasa por el modelo para su predicciòn predecir la clase (1 o -1).

Finalmente para saber a que clase pertenece el nuevo vector $X_j$ vasta con verificar el signo de f(X_j). 

  - **If $f(X_j) >=0$ then $Y_j$ = 1 else $Y_j = -1$**

In [73]:
def Test(X, Y, W, b):
  F_x = np.dot(X, np.transpose(W)) + b
  n, = Y.shape

  Yreturn = np.arange(n)
  for i in range(n):
    if F_x[i] >= 0:
      Yreturn[i] = 1
    else:
      Yreturn[i] = -1
  
  corrects = 0
  for i in range(n):
    if Y[i] == Yreturn[i]:
      corrects += 1
  print(f"correctly classified: {corrects}")
  print(f"incorrectly classified: {n - corrects}")


  # kernel -> producto punto, gaussiano
  

In [88]:
# # https://xavierbourretsicotte.github.io/SVM_implementation.html
lambd = GetLambda(trainX, trainY)

#w parameter in vectorized form
w = ((trainY * lambd).T @ trainX).reshape(-1,1)

#Selecting the set of indices S corresponding to non zero parameters
S = (lambd > 1e-4).flatten()
# print(S)
# print(trainX[S], trainY[S], np.dot(trainX[S], w))

#Computing b
b = np.mean(trainY[S] - np.dot(trainX[S], w))
#Display results
print('lambd = ',lambd[lambd > 1e-4])
print('w = ', w.flatten())
print('b = ', b)

lambd =  [0.7434839  0.74349442]
w =  [-3.15769337e-06  5.94796592e-01 -9.66540646e-01 -4.46099812e-01]
b =  0.9033563558140709


In [93]:
Test(testX, testY, w.T, b)

correctly classified: 30
incorrectly classified: 0


Base de Datos para Las pruebas:
[Download](https://www.google.com/url?sa=t&rct=j&q=&esrc=s&source=web&cd=&ved=2ahUKEwihhZv-9JH3AhWUI7kGHZWDBV4QFnoECEMQAQ&url=http%3A%2F%2Fwww.saedsayad.com%2Fdatasets%2FIris.xls&usg=AOvVaw3HOrA0X468Juw2u4WM-YvO)

En esta base de datos existen 3 clases, solo utilize dos clases para hacer las pruebas.

- Separe el dataset en 70% para entrenar y 30% para hacer las pruebas

- Añada un valor 1 para la primera clase  y  -1 para la segunda clase.

- En la etapa de test, encuentre el número de elementos correctamente clasificados y el número de elementos incorrectamente clasificados para cada clase.

- Cree una matriz de confusión el cual nos mostrará la eficiencia del método.

In [86]:
lambd = GetLambda(trainX, trainY)
I = (lambd > 1e-4).flatten()
w = Get_W(trainX[I], trainY[I], lambd[I])
b = Get_b(trainX[I], w)

print(I, w, b, sep='\n\n')
Test(testX, testY, w, b)

[False False False False False False False False False False False False
 False False False False False False False False False  True False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False  True]

[-4.44089210e-16  1.18958266e+00 -1.93307182e+00 -8.92186993e-01]

0.903339330575488
correctly classified: 30
incorrectly classified: 0
